In [0]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [0]:
source_data = [(1, 'A'), (2, 'B'), (3, 'C'), (4, 'D')]
source_columns = ['id','name']
target_data = [(1, 'A'), (2, 'B'), (4, 'X'), (5, 'F')]
target_columns = ['id','name']

source_df = spark.createDataFrame(data=source_data, schema=source_columns)
target_df = spark.createDataFrame(data=target_data, schema=target_columns)

In [0]:
merged_df = source_df.join(target_df, how = 'full', on ='id')
merged_df = merged_df.withColumn("source_name",source_df.name)\
            .withColumn("target_name",target_df.name)\
            .select("id","source_name","target_name")
merged_df.show()

In [0]:
df = (
    merged_df
    .withColumn(
        "comment",
        F.when(F.col("source_name").isNull(), "New in target")
         .when(F.col("target_name").isNull(), "New in source")
         .when(F.col("source_name") != F.col("target_name"), "misMatch")
    )
    .select("id", "comment")
    .filter(F.col("comment").isNotNull())
)

df.show()

M2

In [0]:
from pyspark.sql.functions import col, when, coalesce

# Full outer join
df = (
    source_df.alias("t1")
    .join(
        target_df.alias("t2"),
        on=(col("t1.id") == col("t2.id")),
        how="full"
    )
)

# Rename columns
df = (
    df.select(
        col("t1.id").alias("source_id"),
        col("t2.id").alias("target_id"),
        col("t1.name").alias("source_name"),
        col("t2.name").alias("target_name")
    )
)

# Add comment column
df = (
    df.withColumn(
        "comment",
        when(
            (col("source_id") == col("target_id")) &
            (col("source_name") != col("target_name")),
            "MisMatched"
        )
        .when(col("target_id").isNull(), "new in source")
        .when(col("source_id").isNull(), "new in target")
    )
)

# Keep only mismatched/new records
df = df.filter(col("comment").isNotNull())

# Create final id column
df = (
    df.withColumn(
        "id",
        coalesce(col("source_id"), col("target_id"))
    )
    .select("id", "comment")
)

df.show()

Q2 Word Count

In [0]:
data = [('python bootcamp1.txt','python for data analytics 0 to hero bootcamp starting on Jan 6th')
,('python bootcamp2.txt','classes will be held on weekends from 11am to 1 pm for 5-6 weeks')
,('python bootcamp3.txt','use code NY2024 to get 33 percent off. You can register from namaste sql website. Link in pinned comment')]

schema = ["filename" , "content"]

df = spark.createDataFrame(data = data , schema = schema)

In [0]:
display(df)

In [0]:
df = df.select("content",F.split(F.col("content")," ").alias("word_array"))
display(df)

In [0]:
df = df.select("content", F.explode(F.col("word_array")).alias("word"))

final_df = (
    df.groupby("word")
        .agg(
            F.count("word").alias("word_count")
        )
        .filter(F.col("word_count") > 1)
)
display(final_df)

Q3

In [0]:
data = [
    (1,),
    (3,),
    (5,),
    (6,),
    (7,),
    (9,),
    (10,)
]
schema = ["id"]
df = spark.createDataFrame(data, schema)

In [0]:
min_no = df.select(F.min("id")).collect()[0][0]
max_no = df.select(F.max("id")).collect()[0][0]
full_df = spark.range(min_no,max_no+1)

merged_df = (
    full_df.join(df, how ='left_anti', on ='id')
)
display(merged_df)